### Read from a PDF 

Read the PDF and create chunks from it.
Embed these chunks and store in a vector store.
Search vector store for the user query using similarity searches.
Feed this to LLM as additional context and let it answer the query.

In [8]:
from pypdf import PdfReader

def load_pdf_text(file_path: str):
    """Extracts text from a PDF file using native pypdf."""
    reader = PdfReader(file_path)
    full_text = []
    page_counter = 0
    
    for page in reader.pages:
        text = page.extract_text()
        if text:
            full_text.append(text)
            page_counter += 1
            
    return "\n".join(full_text), page_counter

# Usage
pdf_text, page_counter = load_pdf_text("documents/bgita.pdf")
print("Total number of pages in the pdf is : " + str(page_counter))  # Print total number of pages
print("Total document length from the pdf is : " + str(len(pdf_text)))  # Print first 500 characters

Total number of pages in the pdf is : 154
Total document length from the pdf is : 315966


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200 )
documents = text_splitter.split_text(pdf_text)
print("Total number of chunks created from the pdf is : " + str(len(documents)))

ids = [f"page_{i}" for i in range(len(documents))]

Total number of chunks created from the pdf is : 397


### Use chromadb to index these chunks and place the embedded vectors.

In [30]:
import chromadb
# 1. Initialize the Chroma client (In-memory by default)
# Use PersistentClient(path="./data") for data persistence
chroma_client = chromadb.PersistentClient(path="./data")

# 2. Create or get a collection
# A collection is where you store your embeddings and documents
collection = chroma_client.get_or_create_collection(name="gita_collection")

# 3. Add documents to the collection
# Chroma automatically generates embeddings if no embedding function is provided
collection.add(
    documents=documents,
    ids=ids
)

In [31]:
query  = "why is arjuna sad in the beginning of the gita ?"
# 4. Perform a similarity search
results = collection.query(
    query_texts=[query], # Your search query
    n_results=5                            # Number of similar results to return
)

print(results["ids"])  # Print the IDs of the similar documents


[['page_14', 'page_34', 'page_268', 'page_291', 'page_93']]


In [45]:
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma 
from langchain_classic.chains import ConversationalRetrievalChain

vector_store = Chroma(
    collection_name="gita_collection",
    persist_directory="./data"
)

def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 5 similar documents
    retriever = vector_store.as_retriever(collection_name="gita_collection", search_type="similarity", search_kwargs={"k": 5})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = qa_chain.invoke({"question": query,"chat_history":[]})
    print(result["answer"])
    matching_docs = result["source_documents"]
    print("Matching document IDs:")
    for doc in matching_docs:
        print(doc.id)

In [ ]:
query = "What is the nature of the soul?" 
query_after_getting_matched_documents(query)


According to the Bhagavad Gita, the nature of the soul (Purusha) is described as eternal, unchanging, and distinct from the material body or matter (Prakriti). The soul is regarded as the true self or Atman, which is immortal and resides within every living being. It is not subject to birth or death like the physical body and is beyond all worldly attachments and desires. The Gita emphasizes that knowing this eternal nature of the soul leads to liberation (moksha) from the cycle of rebirth and suffering.
Matching document IDs:
page_312
page_302
page_303
page_304
page_108


In [48]:
query = "How should one perform karma" 
query_after_getting_matched_documents(query)

According to the teachings in the Bhagavad Gita and the provided commentary, one should perform karma (actions) without attachment or expectation of rewards. The key points are:

1. **Perform Actions with Detachment**: Engage in your legitimate duties but do so without attachment to the results (karmaphalahetur bhoor maa te sango’stwakarmani).

2. **Renounce Egoism and Selfishness**: True renunciation involves giving up selfishness and attachment while performing one’s duties, known as Sattwic Tyaga.

3. **Righteous Intentions**: Actions should be performed for their own sake, not for the fruits or rewards (Karmanyevaadhikaaraste maa phaleshu kadaachana).

4. **Balance in Success and Failure**: Maintain evenness of mind regardless of success or failure (Siddhyasiddhyoh samo bhootwaa samatwam yoga uchyate).

5. **Seek Wisdom Over Action**: Far lower than the Yoga of wisdom is action; seek refuge in wisdom rather than attachment to results (Far lower than the Yoga of wisdom is action, O 